In [ ]:
import os
import cv2
import mediapipe as mp
import splitfolders

mp_face = mp.solutions.face_detection
face_detector = mp_face.FaceDetection(min_detection_confidence=0.5)

TARGET_EMOTION = {
    ord('1'): "Fear",
    ord('2'): "Surprise",
    ord('3'): "Disgust",
    ord('4'): "Sadness",
    ord('5'): "Anger"
}

def buat_folder(output_folder):
    os.makedirs(output_folder, exist_ok=True)
    for folder in TARGET_EMOTION.values():
        os.makedirs(os.path.join(output_folder, folder), exist_ok=True)

def preprocessing_wajah(image):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hasil = face_detector.process(rgb)
    
    if not hasil.detections:
        return None
        
    tinggi, lebar, _ = image.shape
    wajah_terbesar = None
    luas_maksimum = 0
    
    for deteksi in hasil.detections:
        bbox = deteksi.location_data.relative_bounding_box
        luas = bbox.width * bbox.height
        if luas > luas_maksimum:
            luas_maksimum = luas
            wajah_terbesar = bbox
            
    x = max(0, int(wajah_terbesar.xmin * lebar))
    y = max(0, int(wajah_terbesar.ymin * tinggi))
    w = min(lebar - x, int(wajah_terbesar.width * lebar))
    h = min(tinggi - y, int(wajah_terbesar.height * tinggi))
    
    wajah = image[y:y+h, x:x+w]
    
    if wajah.size == 0:
        return None
        
    wajah_resized = cv2.resize(wajah, (224, 224))
    
    yuv = cv2.cvtColor(wajah_resized, cv2.COLOR_BGR2YUV)
    yuv[:,:,0] = cv2.equalizeHist(yuv[:,:,0])
    return cv2.cvtColor(yuv, cv2.COLOR_YUV2BGR)

def proses_dataset_manual(input_folder, output_folder):
    for root, _, files in os.walk(input_folder):
        nama_video = os.path.basename(root)
        for file in files:
            if not file.lower().endswith((".jpg", ".png")):
                continue
                
            gambar = cv2.imread(os.path.join(root, file))
            if gambar is None:
                continue
                
            wajah = preprocessing_wajah(gambar)
            if wajah is None:
                continue

            cv2.imshow("Anotasi FACS | 1:Fear 2:Surprise 3:Disgust 4:Sadness 5:Anger | 0:Skip | ESC:Keluar", wajah)
            key = cv2.waitKey(0)
            
            if key in TARGET_EMOTION:
                emosi = TARGET_EMOTION[key]
                lokasi = os.path.join(output_folder, emosi, f"{nama_video}_{file}")
                cv2.imwrite(lokasi, wajah)
            elif key == 27: 
                cv2.destroyAllWindows()
                return
                
    cv2.destroyAllWindows()

def jalankan_pipeline_total(input_base="dataset_frame", labeled_base="dataset_labeled", final_base="dataset_final"):
    buat_folder(labeled_base)
    proses_dataset_manual(input_base, labeled_base)
    splitfolders.ratio(labeled_base, output=final_base, seed=42, ratio=(0.7, 0.2, 0.1), group_prefix=None, move=False)

if __name__ == "__main__":
    jalankan_pipeline_total()